# 01 — Landsat scene inventory

This notebook builds and validates the Landsat scene inventory used in the Barranquilla SUHI deep learning workflow.

The purpose of this notebook is not to download imagery. Its purpose is to create a controlled, auditable inventory of the Landsat scenes retained for preprocessing, tensor construction, and model training.

## Expected outputs

- A standardized scene inventory table.
- Basic temporal and sensor summaries.
- A validation report indicating missing or inconsistent metadata.
- A template CSV file when no inventory is available yet.

## Reproducibility warning

The scene inventory is a methodological object. Any scene excluded from the workflow must have an explicit exclusion reason. Otherwise, the temporal dataset becomes opaque and difficult to audit.

## 1. Imports

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt

## 2. Project paths

This notebook avoids hard-coded personal paths. It first tries to read `configs/paths.yml`. If that file does not exist, it falls back to `configs/paths_example.yml`.

For a real execution, copy `configs/paths_example.yml` as `configs/paths.yml` and adapt it to your local or cloud environment.

In [ ]:
PROJECT_ROOT = Path.cwd()

# If the notebook is executed from the notebooks/ folder, move one level up.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

paths_file = CONFIG_DIR / "paths.yml"
paths_example_file = CONFIG_DIR / "paths_example.yml"

if paths_file.exists():
    active_paths_file = paths_file
elif paths_example_file.exists():
    active_paths_file = paths_example_file
else:
    active_paths_file = None

if active_paths_file is not None:
    with open(active_paths_file, "r", encoding="utf-8") as f:
        paths_config = yaml.safe_load(f)
else:
    paths_config = {}

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Active path configuration: {active_paths_file}")

## 3. Inventory schema

The inventory must use a fixed schema. This prevents ambiguous scene selection and makes the dataset auditable.

In [ ]:
INVENTORY_COLUMNS = [
    "scene_id",
    "product_id",
    "landsat_mission",
    "sensor",
    "acquisition_date",
    "year",
    "month",
    "path",
    "row",
    "collection",
    "tier",
    "processing_level",
    "cloud_cover",
    "include",
    "exclusion_reason",
    "notes",
]

COLUMN_DESCRIPTIONS = {
    "scene_id": "Short Landsat scene identifier.",
    "product_id": "Full product identifier when available.",
    "landsat_mission": "Landsat mission, e.g., Landsat 5, Landsat 7, Landsat 8, Landsat 9.",
    "sensor": "Sensor name, e.g., TM, ETM+, OLI/TIRS, OLI-2/TIRS-2.",
    "acquisition_date": "Image acquisition date in YYYY-MM-DD format.",
    "year": "Acquisition year.",
    "month": "Acquisition month.",
    "path": "WRS-2 path.",
    "row": "WRS-2 row.",
    "collection": "Landsat collection, e.g., Collection 2.",
    "tier": "Landsat tier, e.g., T1 or T2.",
    "processing_level": "Processing level, e.g., L2SP.",
    "cloud_cover": "Scene-level cloud-cover percentage.",
    "include": "Boolean flag indicating whether the scene is retained.",
    "exclusion_reason": "Reason for exclusion when include is False.",
    "notes": "Additional comments.",
}

pd.DataFrame(
    [{"column": k, "description": v} for k, v in COLUMN_DESCRIPTIONS.items()]
)

## 4. Create or load the inventory table

If `data/landsat_scene_inventory.csv` does not exist, this notebook creates a template file. Fill the template before running the validation steps with the final scene inventory.

In [ ]:
inventory_path = DATA_DIR / "landsat_scene_inventory.csv"

def create_inventory_template(path: Path) -> pd.DataFrame:
    template = pd.DataFrame(columns=INVENTORY_COLUMNS)
    template.to_csv(path, index=False, encoding="utf-8")
    return template

if inventory_path.exists():
    inventory = pd.read_csv(inventory_path)
    print(f"Loaded inventory: {inventory_path}")
else:
    inventory = create_inventory_template(inventory_path)
    print(f"Inventory template created: {inventory_path}")

inventory.head()

## 5. Basic schema validation

In [ ]:
def validate_inventory_schema(df: pd.DataFrame, expected_columns: list[str]) -> dict:
    current_columns = list(df.columns)
    missing_columns = [col for col in expected_columns if col not in current_columns]
    extra_columns = [col for col in current_columns if col not in expected_columns]
    return {
        "n_rows": len(df),
        "n_columns": len(df.columns),
        "missing_columns": missing_columns,
        "extra_columns": extra_columns,
        "schema_is_valid": len(missing_columns) == 0,
    }

schema_report = validate_inventory_schema(inventory, INVENTORY_COLUMNS)
schema_report

## 6. Data type harmonization

The following cell standardizes dates, year, month, cloud cover, path, row, and inclusion flags.

In [ ]:
def harmonize_inventory(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in INVENTORY_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    df = df[INVENTORY_COLUMNS]

    if len(df) == 0:
        return df

    df["acquisition_date"] = pd.to_datetime(df["acquisition_date"], errors="coerce")
    df["year"] = df["acquisition_date"].dt.year
    df["month"] = df["acquisition_date"].dt.month

    for col in ["path", "row"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    df["cloud_cover"] = pd.to_numeric(df["cloud_cover"], errors="coerce")

    if "include" in df.columns:
        df["include"] = df["include"].replace({"TRUE": True, "FALSE": False, "True": True, "False": False})
        df["include"] = df["include"].astype("boolean")

    text_columns = [
        "scene_id", "product_id", "landsat_mission", "sensor", "collection",
        "tier", "processing_level", "exclusion_reason", "notes"
    ]
    for col in text_columns:
        df[col] = df[col].astype("string")

    return df

inventory = harmonize_inventory(inventory)
inventory.head()

## 7. Inventory quality checks

This section identifies missing metadata, duplicated scenes, invalid dates, and retained scenes without required information.

In [ ]:
def inventory_quality_checks(df: pd.DataFrame) -> pd.DataFrame:
    checks = []

    required_for_all = ["scene_id", "landsat_mission", "sensor", "acquisition_date", "path", "row"]
    required_for_included = ["scene_id", "acquisition_date", "path", "row", "cloud_cover"]

    for col in required_for_all:
        checks.append({
            "check": f"missing_{col}",
            "count": int(df[col].isna().sum()) if col in df.columns else len(df),
            "severity": "high",
        })

    if "scene_id" in df.columns:
        checks.append({
            "check": "duplicated_scene_id",
            "count": int(df["scene_id"].duplicated().sum()),
            "severity": "high",
        })

    if len(df) > 0 and "include" in df.columns:
        included = df[df["include"] == True].copy()
        for col in required_for_included:
            checks.append({
                "check": f"included_missing_{col}",
                "count": int(included[col].isna().sum()) if col in included.columns else len(included),
                "severity": "high",
            })

        excluded_without_reason = df[(df["include"] == False) & (df["exclusion_reason"].isna() | (df["exclusion_reason"].astype(str).str.strip() == ""))]
        checks.append({
            "check": "excluded_without_reason",
            "count": int(len(excluded_without_reason)),
            "severity": "medium",
        })

    return pd.DataFrame(checks)

quality_report = inventory_quality_checks(inventory)
quality_report

## 8. Inventory summaries

These summaries should be inspected before preprocessing begins.

In [ ]:
if len(inventory) == 0:
    print("The inventory is empty. Fill data/landsat_scene_inventory.csv and rerun this notebook.")
else:
    included_inventory = inventory[inventory["include"] == True].copy()

    summary = {
        "total_scenes": int(len(inventory)),
        "included_scenes": int(len(included_inventory)),
        "excluded_scenes": int(len(inventory) - len(included_inventory)),
        "first_date": str(included_inventory["acquisition_date"].min().date()) if len(included_inventory) else None,
        "last_date": str(included_inventory["acquisition_date"].max().date()) if len(included_inventory) else None,
        "missions": sorted(included_inventory["landsat_mission"].dropna().unique().tolist()),
        "sensors": sorted(included_inventory["sensor"].dropna().unique().tolist()),
    }
    summary

## 9. Temporal distribution plots

The plots below are diagnostic only. They help detect temporal gaps, sensor imbalance, and irregular scene distribution.

In [ ]:
if len(inventory) > 0:
    included_inventory = inventory[inventory["include"] == True].copy()

    if len(included_inventory) > 0:
        scenes_by_year = included_inventory.groupby("year").size().sort_index()

        fig, ax = plt.subplots(figsize=(10, 4))
        scenes_by_year.plot(kind="bar", ax=ax)
        ax.set_title("Included Landsat scenes by year")
        ax.set_xlabel("Year")
        ax.set_ylabel("Number of scenes")
        plt.tight_layout()
        plt.show()
    else:
        print("No included scenes available for plotting.")
else:
    print("Inventory is empty.")

In [ ]:
if len(inventory) > 0:
    included_inventory = inventory[inventory["include"] == True].copy()

    if len(included_inventory) > 0:
        scenes_by_sensor = included_inventory.groupby("sensor").size().sort_values(ascending=False)

        fig, ax = plt.subplots(figsize=(7, 4))
        scenes_by_sensor.plot(kind="bar", ax=ax)
        ax.set_title("Included Landsat scenes by sensor")
        ax.set_xlabel("Sensor")
        ax.set_ylabel("Number of scenes")
        plt.tight_layout()
        plt.show()
    else:
        print("No included scenes available for plotting.")
else:
    print("Inventory is empty.")

## 10. Save standardized inventory and reports

In [ ]:
standardized_inventory_path = OUTPUT_TABLES_DIR / "01_landsat_scene_inventory_standardized.csv"
quality_report_path = OUTPUT_TABLES_DIR / "01_landsat_scene_inventory_quality_report.csv"

inventory.to_csv(standardized_inventory_path, index=False, encoding="utf-8")
quality_report.to_csv(quality_report_path, index=False, encoding="utf-8")

print(f"Standardized inventory saved to: {standardized_inventory_path}")
print(f"Quality report saved to: {quality_report_path}")

## 11. Interpretation checklist

Before moving to preprocessing, verify that:

1. The inventory is not empty.
2. All included scenes have valid acquisition dates.
3. All included scenes have valid path/row metadata.
4. Excluded scenes have explicit exclusion reasons.
5. The temporal distribution is consistent with the study design.
6. The sensor distribution is compatible with the intended cross-sensor analysis.
7. The final included scene count matches the manuscript.

If any of these checks fail, the inventory must be corrected before preprocessing begins.